In [1]:
import os
import glob
import pandas as pd
import config
import numpy as np 

from experiments.iran_market import (load_market_file,build_features)
from utils.detect_price_breaks import detect_price_breaks
from models import get_model
from evaluation.rolling import run_rolling


# ============================================================
# Files
# ============================================================

CSV_FOLDER = r".\data\CSV"

files = sorted(
    glob.glob(os.path.join(CSV_FOLDER, "*.csv"))
)

MODELS = ["ridge", "lasso", "elasticnet"]
THRESHOLDS = [0.20]

# ============================================================
# Run one stock
# ============================================================

def run_one_stock(file_path, threshold):
    config.USE_BACKWARD_ADJUSTMENT = False
    config.ADJUSTMENT_THRESHOLD = threshold
    

    print("\n")
    print("=" * 70)
    print("FILE:", os.path.basename(file_path))
    print("=" * 70)


    results_settings = {}


    for use_adjustment in [False, True]:

        config.USE_BACKWARD_ADJUSTMENT = use_adjustment
        if use_adjustment:
            config.ADJUSTMENT_THRESHOLD = threshold

        print(
            "Adjustment:",
            "ON" if use_adjustment else "OFF"
        )


        df = load_market_file(file_path)

        events_test, _ = detect_price_breaks(df, threshold=threshold)

        print(f"Direct detect_price_breaks - threshold={threshold} - events={len(events_test)}")

        
        X, y, feature_dates, target_dates, n_events = build_features(df)


        X = X.iloc[:config.IRAN_MAX_OBSERVATIONS].reset_index(drop=True)
        y = y.iloc[:config.IRAN_MAX_OBSERVATIONS].reset_index(drop=True)


        rows = []


        for model_name in MODELS:


            model = get_model(model_name)


            results = run_rolling(
                model_module=model,
                X=X,
                y=y,
                train_size=config.IRAN_TRAIN_SIZE,
                step_size=config.IRAN_STEP_SIZE,
                mode=config.ROLLING
            )

            print("TYPE OF RESULTS:", type(results))
            print("RESULTS:", results)

            s = results["summary"]


            rows.append({

                "Model": model_name,
                "R2": s["R2"],
                "R2_OOS": s["R2_OOS"],
                "MSPE": s["MSPE"],
                "RMSE": s["RMSE"],
                "Use_Adjustment": use_adjustment,
                "Adjustment_Events": n_events,

            })


        results_settings[use_adjustment] = pd.DataFrame(rows)


    # ========================================================
    # Compare RAW vs ADJUSTED
    # ========================================================

    raw = results_settings[False]
    adj = results_settings[True]


    comp = raw.copy()


    comp = comp.rename(columns={

        "R2":"R2_raw",
        "R2_OOS":"R2_OOS_raw",
        "MSPE":"MSPE_raw",
        "RMSE":"RMSE_raw",

    })


    comp["R2_adj"] = adj["R2"].values
    comp["R2_OOS_adj"] = adj["R2_OOS"].values
    comp["MSPE_adj"] = adj["MSPE"].values
    comp["RMSE_adj"] = adj["RMSE"].values


    comp["Delta_R2_OOS"] = (
        comp["R2_OOS_adj"] -
        comp["R2_OOS_raw"]
    )

    # تعداد eventهای حالت تعدیل‌شده
    comp["Adjustment_Events"] = adj["Adjustment_Events"].values

    comp["Delta_R2_OOS"] = (comp["R2_OOS_adj"] -comp["R2_OOS_raw"])
    
    comp["Ticker"] = os.path.basename(file_path)

    

    return comp



# ============================================================
# Run all files
# ============================================================

all_results = []

for threshold in THRESHOLDS:

    print("\n")
    print("#"*70)
    print("RUNNING THRESHOLD:", threshold)
    print("#"*70)


    for file in files:

        try:

            result = run_one_stock(
                file,
                threshold
            )

            result["Threshold"] = threshold

            all_results.append(result)


        except Exception as e:

            print(
                "FAILED:",
                os.path.basename(file),
                "|",
                e
            )

# ============================================================
# Final table
# ============================================================

final_table = pd.concat(
    all_results,
    ignore_index=True
)


print("\n")
print("=" * 70)
print("FINAL STOCK RESULTS")
print("=" * 70)


print(
    final_table[
        [
    "Threshold",
    "Ticker",
    "Model",
    "R2_OOS_raw",
    "R2_OOS_adj",
    "Delta_R2_OOS",
    "Adjustment_Events",
        ]
                ]
    .to_string(index=False)
)



# ============================================================
# Market level summary
# ============================================================


print("\n")
print("=" * 70)
print("MARKET LEVEL SUMMARY")
print("=" * 70)



summary = (
    final_table
    .groupby(["Threshold","Model"])
    ["Delta_R2_OOS"]
    .agg(
        Mean="mean",
        Median="median",
        Std="std",
        Count="count"
    )
)


print(summary)


print("\n")


improved = (
    final_table
    .groupby(["Threshold","Model"])
    ["Delta_R2_OOS"]
    .apply(lambda x: (x > 0).mean()*100)
)


print("Percent Improved Stocks (%)")
print(improved)



######################################################################
RUNNING THRESHOLD: 0.2
######################################################################


FILE: S_Mobarakeh Steel.csv
Adjustment: OFF
Reading File:
.\data\CSV\S_Mobarakeh Steel.csv
First date : 2007-03-11 00:00:00
Last date  : 2025-12-21 00:00:00

Columns:
['<TICKER>', '<DTYYYYMMDD>', '<FIRST>', '<HIGH>', '<LOW>', '<CLOSE>', '<VALUE>', '<VOL>', '<OPENINT>', '<PER>', '<OPEN>', '<LAST>']

First Row:
<TICKER>          S*Mobarakeh Steel
<DTYYYYMMDD>    2007-03-11 00:00:00
<FIRST>                      1900.0
<HIGH>                       1938.0
<LOW>                        1900.0
<CLOSE>                      1900.0
<VALUE>              889437216900.0
<VOL>                     468077431
<OPENINT>                      7736
<PER>                             D
<OPEN>                       1900.0
<LAST>                       1900.0
Name: 0, dtype: object

DataFrame Shape:
(4468, 12)
Direct detect_price_breaks - thresho

Rolling windows: 460it [00:23, 19.75it/s]                         


TYPE OF RESULTS: <class 'dict'>
RESULTS: {'summary': {'R2': -0.005311085008400829, 'R2_OOS': np.float64(-0.004097927744655783), 'MSPE': 0.0011887189331252074, 'RMSE': np.float64(0.03447780348463642)}, 'results':        Target  Prediction
0    0.000000   -0.011521
1   -0.009848   -0.008529
2   -0.030381   -0.007656
3   -0.025336   -0.006609
4   -0.024207   -0.007162
..        ...         ...
445  0.007692    0.000522
446 -0.034695   -0.003224
447 -0.012373    0.001112
448  0.004008    0.003358
449  0.001199    0.002369

[450 rows x 2 columns], 'parameters': {}, 'windows': 23}


Rolling windows: 460it [00:02, 217.95it/s]                         


TYPE OF RESULTS: <class 'dict'>
RESULTS: {'summary': {'R2': -0.32477672635511734, 'R2_OOS': np.float64(-0.32317805452867), 'MSPE': 0.0015664675345431014, 'RMSE': np.float64(0.03957862471768191)}, 'results':        Target  Prediction
0    0.000000   -0.022500
1   -0.009848   -0.015136
2   -0.030381   -0.012214
3   -0.025336   -0.006540
4   -0.024207   -0.006376
..        ...         ...
445  0.007692   -0.000917
446 -0.034695   -0.002043
447 -0.012373    0.000384
448  0.004008    0.001796
449  0.001199    0.000812

[450 rows x 2 columns], 'parameters': {}, 'windows': 23}


...

TYPE OF RESULTS: <class 'dict'>
RESULTS: {'summary': {'R2': 0.010105495543218934, 'R2_OOS': np.float64(0.010500312157176506), 'MSPE': 0.000496708895671669, 'RMSE': np.float64(0.0222869669464391)}, 'results':        Target  Prediction
0    0.000000   -0.011521
1   -0.009848   -0.008529
2   -0.030381   -0.007656
3   -0.025336   -0.006609
4   -0.024207   -0.007162
..        ...         ...
445  0.007692    0.000310
446 -0.034695   -0.001294
447 -0.012373   -0.001712
448  0.004008   -0.000333
449  0.001199    0.000009

[450 rows x 2 columns], 'parameters': {}, 'windows': 23}


Rolling windows: 460it [00:01, 248.03it/s]                         


TYPE OF RESULTS: <class 'dict'>
RESULTS: {'summary': {'R2': -0.0019082966431134274, 'R2_OOS': np.float64(-0.0015086883622352776), 'MSPE': 0.0005027371718393165, 'RMSE': np.float64(0.022421801262149224)}, 'results':        Target  Prediction
0    0.000000   -0.022500
1   -0.009848   -0.015136
2   -0.030381   -0.012214
3   -0.025336   -0.006540
4   -0.024207   -0.006376
..        ...         ...
445  0.007692    0.000054
446 -0.034695   -0.001749
447 -0.012373   -0.000908
448  0.004008   -0.000621
449  0.001199    0.000701

[450 rows x 2 columns], 'parameters': {}, 'windows': 23}


Rolling windows: 460it [00:06, 71.40it/s]                         

TYPE OF RESULTS: <class 'dict'>
RESULTS: {'summary': {'R2': -0.0002034734416140882, 'R2_OOS': np.float64(0.00019545487536809514), 'MSPE': 0.0005018817262883819, 'RMSE': np.float64(0.022402716939879903)}, 'results':        Target  Prediction
0    0.000000   -0.022394
1   -0.009848   -0.015062
2   -0.030381   -0.012118
3   -0.025336   -0.006454
4   -0.024207   -0.006294
..        ...         ...
445  0.007692   -0.000104
446 -0.034695   -0.002041
447 -0.012373   -0.001172
448  0.004008   -0.000866
449  0.001199    0.000533

[450 rows x 2 columns], 'parameters': {}, 'windows': 23}


FINAL STOCK RESULTS
 Threshold                Ticker      Model  R2_OOS_raw  R2_OOS_adj  Delta_R2_OOS  Adjustment_Events
       0.2 S_Mobarakeh Steel.csv      ridge   -0.004098    0.010500      0.014598                 13
       0.2 S_Mobarakeh Steel.csv      lasso   -0.323178   -0.001509      0.321669                 13
       0.2 S_Mobarakeh Steel.csv elasticnet   -0.323954    0.000195      0.324149         

In [2]:
df = load_market_file(r".\data\CSV\S_Mobarakeh Steel.csv")

events, data = detect_price_breaks(
    df,
    threshold=0.20
)

print(
    events[
        [
            "<DTYYYYMMDD>",
            "prev_last",
            "<FIRST>",
            "<LAST>",
            "raw_return",
            "factor"
        ]
    ].to_string(index=False)
)

Reading File:
.\data\CSV\S_Mobarakeh Steel.csv
First date : 2007-03-11 00:00:00
Last date  : 2025-12-21 00:00:00

Columns:
['<TICKER>', '<DTYYYYMMDD>', '<FIRST>', '<HIGH>', '<LOW>', '<CLOSE>', '<VALUE>', '<VOL>', '<OPENINT>', '<PER>', '<OPEN>', '<LAST>']

First Row:
<TICKER>          S*Mobarakeh Steel
<DTYYYYMMDD>    2007-03-11 00:00:00
<FIRST>                      1900.0
<HIGH>                       1938.0
<LOW>                        1900.0
<CLOSE>                      1900.0
<VALUE>              889437216900.0
<VOL>                     468077431
<OPENINT>                      7736
<PER>                             D
<OPEN>                       1900.0
<LAST>                       1900.0
Name: 0, dtype: object

DataFrame Shape:
(4468, 12)
<DTYYYYMMDD>  prev_last  <FIRST>  <LAST>  raw_return   factor
  2008-10-26     3689.0   2200.0  2242.0   -0.516898 0.596368
  2009-06-23     2122.0   1699.0  1662.0   -0.222319 0.800660
  2011-02-19     3244.0   2200.0  2190.0   -0.388350 0.678175
 